In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 566.8 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=5799d1420920347c4ebe3f242494285fc5c19122dcfab581acd685995a52f5ce
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [2]:
# Generate a list of random bits

def generate_random_bits(length):
  qc = QuantumCircuit(length, length)
  for i in range(length):
    qc.h(i)
  qc.measure(range(length), range(length))

  simulator = BasicSimulator()
  compiled_circuit = transpile(qc, simulator)
  job = simulator.run(compiled_circuit, shots=1)

  result_string = list(job.result().get_counts().keys())[0]
  result_string = result_string[::-1]

  return [int(bit) for bit in result_string]

In [7]:
# Agents - Alice & Bob & Eve (attacker)

# alice does the encoding while bob will try to measure / guess.
def alice_encode(bits, bases):
  num_qubits = len(bits)
  qc = QuantumCircuit(num_qubits, num_qubits)

  # if bit is 1, apply X to make it 1|>
  # if basis is 1, apply H.
  for i in range(num_qubits):
      if bits[i] == 1:
          qc.x(i)
      if bases[i] == 1:
          qc.h(i)
  return qc

# bob measures and guesses.
def bob_measure(qc, bases):
    num_qubits = len(bases)

    # apply H if basis is 1 to rotate back towards standard basis for measuring
    for i in range(num_qubits):
        if bases[i] == 1:
            qc.h(i)
        qc.measure(i, i)

    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)

    result_string = list(job.result().get_counts().keys())[0]
    result_string = result_string[::-1]

    return [int(bit) for bit in result_string]

# eve is here to intercept :(
def eve_intercept(quantum_channel, num_qubits):
    eve_guessed_bases = generate_random_bits(num_qubits)
    eve_measured_bits = bob_measure(quantum_channel, eve_guessed_bases)
    madeup_quantum_channel = alice_encode(eve_measured_bits, eve_guessed_bases)

    return madeup_quantum_channel, eve_measured_bits, eve_guessed_bases

In [8]:
# generate bits and bases to start the simulation:

# we will eventually send a letter to determine if the decryption works with the key.
NUM_QUBITS = 24

alice_secret_bits = generate_random_bits(NUM_QUBITS)
alice_secret_bases = generate_random_bits(NUM_QUBITS)
bob_guessed_bases = generate_random_bits(NUM_QUBITS)

print(f"Alice's Bits:  {alice_secret_bits}")
print(f"Alice's Bases: {alice_secret_bases} (0 = +, 1 = x)")
print(f"Bob's Bases:   {bob_guessed_bases} (0 = +, 1 = x)")

Alice's Bits:  [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0]
Alice's Bases: [0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1] (0 = +, 1 = x)
Bob's Bases:   [0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1] (0 = +, 1 = x)


In [9]:
# alice sends them, bob will measure.
quantum_channel = alice_encode(alice_secret_bits, alice_secret_bases)

# eve replaces the quantum channel with their made up forged channel
madeup_quantum_channel, eve_bits, eve_bases = eve_intercept(quantum_channel, NUM_QUBITS)

# Bob intercepts and measures
bob_measured_bits = bob_measure(madeup_quantum_channel, bob_guessed_bases)
print(f"Bob's Results: {bob_measured_bits}")

Bob's Results: [0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0]


In [10]:
# compare now to determine if their keys match:
alice_key = []
bob_key = []

for i in range(NUM_QUBITS):
    if alice_secret_bases[i] == bob_guessed_bases[i]:
        alice_key.append(alice_secret_bits[i])
        bob_key.append(bob_measured_bits[i])

print("--- FINAL RESULTS ---")
print(f"Alice's Key: {alice_key}")
print(f"Bob's Key:   {bob_key}")

# we check for errors to see if the keys match.
errors = 0
for i in range(len(alice_key)):
    if alice_key[i] != bob_key[i]:
        errors += 1

error_rate = (errors / len(alice_key)) * 100 if len(alice_key) > 0 else 0

print(f"Errors Detected: {errors}")
print(f"Error Rate:      {error_rate:.2f}%")

if error_rate > 0:
    print("\nEve intercepted. Not safe.")

else:
    print("\nNo errors detected.")

    # we can start to encrypt here.
    message = "Q"
    message_bits = []
    for char in message:
        binary_char = format(ord(char), '08b')
        for bit in binary_char:
            message_bits.append(int(bit))

    print(f"Original Message: '{message}'")
    print(f"Message Length:   {len(message_bits)} bits")

    # check if the message bits length is more than the key. if it is, we can't encrypt it and have to re-run a new set.
    if len(alice_key) < len(message_bits):
      print(f"[ERROR] The length of message bits {len(message_bits)} is more than the key bits {len(alice_key)}.")

    # start to XOR with the key.
    else:
      # reduce key length down to message bit length;
      alice_pad = alice_key[:len(message_bits)]
      bob_pad = bob_key[:len(message_bits)]

      ciphertext = []
      for i in range(len(message_bits)):
        scrambled_bit = message_bits[i] ^ alice_pad[i]
        ciphertext.append(scrambled_bit)

      print(f"\nAlice's Key:    {alice_pad}")
      print(f"Alice's Message:  {message_bits}")
      print(f"Alice encoded message:  {ciphertext}")

      # now decode on Bob's side with XOR again;
      decrypted_bits = []
      for i in range(len(ciphertext)):
        unscrambled_bit = ciphertext[i] ^ bob_pad[i]
        decrypted_bits.append(unscrambled_bit)

      # then convert back to a character:
      decrypted = ""
      for i in range(0, len(decrypted_bits), 8):
        byte = decrypted_bits[i:i+8]
        byte_string = "".join(str(bit) for bit in byte)
        decrypted += chr(int(byte_string, 2))

        print(f"\nBob receives:     {ciphertext} (Ciphertext)")
        print(f"Bob's Key:        {bob_pad}")
        print(f"Bob Decrypts:     {decrypted_bits}")
        print(f"\nReceived value: '{decrypted}'")




--- FINAL RESULTS ---
Alice's Key: [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0]
Bob's Key:   [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]
Errors Detected: 1
Error Rate:      9.09%

Eve intercepted. Not safe.


When Eve intercepts, it changes Bob's key. We can compare Alice's and Bob's key to determine if there are any mismatch. If there is one, that means Eve has intercepted, as Eve is required to send a 'made up' quantum channel back to Bob.

While its possible that Eve intercepts without Bob finding out (although highly unlikely), there is a 75% chance for Eve to get lucky to not be detected.

But if 10 bits are checked, the probability would be (0.75)^10 for Eve to not be detected, which would be super unlikely. This more the bits, the lower the probability.
